In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load the features dataset
df = pd.read_csv("data/har_features.csv")

# 2. Create the binary target variable (Case-insensitive check for "scroll")
df["is_scrolling"] = df["label"].str.contains("scroll", case=False, na=False).astype(int)

print(f"Total dataset size: {len(df)} windows")
print(f"Overall target distribution:\n{df['is_scrolling'].value_counts(normalize=True)}\n")

# 3. Separate Features, Target, and Groups
X = df.drop(columns=["session_id", "window_start_idx", "label", "is_scrolling"])
y = df["is_scrolling"]
groups = df["session_id"]  

# 4. Stratified Group-Based Validation Split
# Ensures sessions don't leak, but forces a balanced target mix in train/test sets
sgkf = StratifiedGroupKFold(n_splits=4)
train_idx, test_idx = next(sgkf.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Print out the balance verification to check the fix
print("=== Split Verification ===")
print(f"Training on {len(X_train)} windows from {groups.iloc[train_idx].nunique()} sessions.")
print(f"Train target balance:\n{y_train.value_counts(normalize=True).to_string()}\n")

print(f"Testing on {len(X_test)} windows from {groups.iloc[test_idx].nunique()} sessions.")
print(f"Test target balance:\n{y_test.value_counts(normalize=True).to_string()}\n")

# 5. Scale Features (Fit only on train data to prevent data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train the Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train_scaled, y_train)

# 7. Evaluate the Model
y_pred = clf.predict(X_test_scaled)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Not Scrolling", "Scrolling"]))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

# 8. Check Feature Importance
importances = pd.Series(clf.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(5)
print("\n=== Top 5 Most Important Features ===")
print(top_features.to_string())

Total dataset size: 997 windows
Overall target distribution:
is_scrolling
1    0.509529
0    0.490471
Name: proportion, dtype: float64

=== Split Verification ===
Training on 649 windows from 5 sessions.
Train target balance:
is_scrolling
0    0.577812
1    0.422188

Testing on 348 windows from 3 sessions.
Test target balance:
is_scrolling
1    0.672414
0    0.327586

=== Classification Report ===
               precision    recall  f1-score   support

Not Scrolling       0.33      1.00      0.49       114
    Scrolling       0.00      0.00      0.00       234

     accuracy                           0.33       348
    macro avg       0.16      0.50      0.25       348
 weighted avg       0.11      0.33      0.16       348

=== Confusion Matrix ===
[[114   0]
 [234   0]]

=== Top 5 Most Important Features ===
gyroscope_y_energy         0.190944
gyroscope_x_std            0.186261
orientation_pitch_std      0.171830
orientation_roll_energy    0.165289
gyroscope_x_energy         0.088203

/home/rems/code/Stat-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/rems/code/Stat-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/rems/code/Stat-Machine-Learning/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave